# Groups and surnames

`PassengerId` splits on the underscore into a group id and a person number. The training file has 6217 groups.

| People in the group | Groups |
|---:|---:|
| 1 | 4805 |
| 2 | 841 |
| 3 | 340 |
| 4 | 103 |
| 5 | 53 |
| 6 | 29 |
| 7 | 33 |
| 8 | 13 |

The group sizes account for every training passenger:

$$
\begin{aligned}
&1\cdot 4805 + 2\cdot 841 + 3\cdot 340 + 4\cdot 103 \\
&\quad + 5\cdot 53 + 6\cdot 29 + 7\cdot 33 + 8\cdot 13 \\
&= 8693.
\end{aligned}
$$

Groups of size at least 2 number $841+340+103+53+29+33+13 = 1412$. In 615 of those groups every member has the same fate. In the other $1412 - 615 = 797$ groups the fates differ. Altark and Solam agree. They are one of the 615, not a law.

Home planet behaves differently. In all 1411 multi-person groups that have at least one filled planet, every filled planet cell is the same planet. A blank planet can therefore be copied from a groupmate when the group has exactly one known planet. That copy fills 90 of the 201 blank planet cells. The other $201 - 90 = 111$ blanks have no unique planet in the group to copy.

Names are two words on all 8493 filled name cells, and 200 names are blank. The last word, the surname, takes 2217 distinct values. Of those, 1835 appear on more than one passenger.

The test file has 3063 group ids. None of them appear in the training file. A rule that copies a training groupmate's fate onto a test passenger in the same group has no such pair to copy.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "13-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# Group sizes, shared fates, shared planets, surnames, and the test groups.
from collections import Counter, defaultdict
groups = defaultdict(list)
for row in train:
    groups[row["PassengerId"].split("_")[0]].append(row)
sizes = Counter(len(members) for members in groups.values())
print(dict(sorted(sizes.items())))
assert dict(sizes) == {1: 4805, 2: 841, 3: 340, 4: 103, 5: 53, 6: 29, 7: 33, 8: 13}
passenger_total = sum(size * count for size, count in sizes.items())
assert passenger_total == 8693
multi = [members for members in groups.values() if len(members) > 1]
same_fate = sum(
    len({row["Transported"] for row in members}) == 1 for members in multi
)
print("multi", len(multi), "same fate", same_fate)
assert len(multi) == 1412
assert same_fate == 615
assert 1412 - same_fate == 797

planet_groups = 0
planet_agreements = 0
filled_blanks = 0
blank_planets = 0
for members in groups.values():
    known = {row["HomePlanet"] for row in members if row["HomePlanet"] != ""}
    blanks = sum(row["HomePlanet"] == "" for row in members)
    blank_planets += blanks
    if len(members) > 1 and known:
        planet_groups += 1
        planet_agreements += len(known) == 1
    if len(known) == 1:
        filled_blanks += blanks
print("planet groups", planet_groups, planet_agreements)
print("blanks", blank_planets, "filled from group", filled_blanks)
assert (planet_groups, planet_agreements) == (1411, 1411)
assert (blank_planets, filled_blanks) == (201, 90)

surnames = [
    row["Name"].split()[-1] for row in train if row["Name"] != ""
]
surname_counts = Counter(surnames)
repeated = sum(count > 1 for count in surname_counts.values())
print(len(surnames), len(surname_counts), repeated)
assert len(surnames) == 8493
assert len(surname_counts) == 2217
assert repeated == 1835
assert all(len(row["Name"].split()) == 2 for row in train if row["Name"] != "")

test_groups = {row["PassengerId"].split("_")[0] for row in test}
train_groups = set(groups)
print("shared groups", len(train_groups & test_groups))
assert len(test_groups) == 3063
assert train_groups.isdisjoint(test_groups)


{1: 4805, 2: 841, 3: 340, 4: 103, 5: 53, 6: 29, 7: 33, 8: 13}
multi 1412 same fate 615
planet groups 1411 1411
blanks 201 filled from group 90
8493 2217 1835
shared groups 0


## Pitfall

Copying `Transported` from one groupmate to another is wrong on the 797 mixed groups. It also cannot fill a test row from a training row, because the two files share 0 group ids. Copying `HomePlanet` is a different operation: the planet really is shared, and the copy fills 90 blanks.

## Summary

6217 training groups, 4805 of them solo, 615 multi-person groups unanimous, 797 mixed. Planet agrees whenever a multi-person group has a planet. Surnames: 2217 distinct, 1835 repeated. Train and test share no group id.
